# Pseudo-Bulk Deconvolution Using the Fusion Signature

Uses the compact gene signature and RF classifier from `Fusion_embedding_classifier_comparison.ipynb`
(Sections 11-13) to build and test a bulk deconvolution pipeline:

1. Split single cells into a **reference pool** (build per-class average expression profiles) and a
   **bulk-simulation pool** (drawn from to build synthetic pseudo-bulk mixtures) -- kept separate so
   deconvolution is never evaluated against the same cells used to build its own reference.
2. Simulate pseudo-bulk samples with random, known mixing fractions of fused/parental/resistant
   cells, combined by averaging.
3. Deconvolve each pseudo-bulk with non-negative least squares (NNLS) against the reference matrix
   to estimate mixing fractions.
4. Cross-check against an independent estimate: run the single-cell RF classifier on the individual
   cells that went into each pseudo-bulk and take the empirical fraction of each predicted label.
5. Compare both estimates against the known true fractions, and against each other.

Needs `fusion_signature_expression.h5ad` and `fusion_signature_classifier.pkl`, both written by the
last cell of Section 13 in `Fusion_embedding_classifier_comparison.ipynb` -- run that notebook
through Section 13 first if these files aren't present yet.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from scipy.optimize import nnls
from scipy.stats import pearsonr

## 1. Load the Fusion Signature Expression and Classifier

In [ ]:
SIGNATURE_EXPRESSION_PATH = 'fusion_signature_expression.h5ad'
SIGNATURE_CLASSIFIER_PATH = 'fusion_signature_classifier.pkl'

sig_adata = ad.read_h5ad(SIGNATURE_EXPRESSION_PATH)
clf_rf_sig = joblib.load(SIGNATURE_CLASSIFIER_PATH)

signature_genes = sig_adata.var_names.tolist()
print(f'{len(signature_genes)} signature genes, {sig_adata.n_obs} cells')
print('Classes:', sorted(sig_adata.obs["sample"].unique()))

## 2. Simulation Parameters

In [ ]:
RANDOM_SEED = 42
REF_POOL_FRACTION = 0.5   # fraction of cells reserved to build per-class reference profiles
N_PSEUDOBULKS = 200
N_CELLS_PER_BULK = 200
DIRICHLET_ALPHA = 1.0     # uniform-over-simplex sampling of mixing fractions across pseudobulks

rng = np.random.default_rng(RANDOM_SEED)
classes = sorted(sig_adata.obs['sample'].unique())
print('Classes (fixed order used throughout):', classes)

## 3. Split Cells into a Reference Pool and a Bulk-Simulation Pool

Stratified by `sample` so both pools stay representative of all three classes. The reference pool
builds the per-class signature matrix used by NNLS; only the bulk-simulation pool is ever sampled
into pseudo-bulks, so deconvolution is never evaluated against its own reference cells.

In [ ]:
ref_idx, bulk_idx = train_test_split(
    np.arange(sig_adata.n_obs),
    test_size=1 - REF_POOL_FRACTION,
    random_state=RANDOM_SEED,
    stratify=sig_adata.obs['sample'].values,
)
print(f'Reference pool: {len(ref_idx)} cells | Bulk-simulation pool: {len(bulk_idx)} cells')

bulk_pool_class_idx = {
    c: bulk_idx[sig_adata.obs['sample'].values[bulk_idx] == c]
    for c in classes
}
for c in classes:
    print(f'  {c}: {len(bulk_pool_class_idx[c])} cells available in the bulk-simulation pool')

## 4. Build the Reference Signature Matrix

NNLS assumes linear mixing (bulk = weighted sum of per-cell profiles), but the exported expression
is `asinh`-transformed -- `asinh` isn't additive across cells (`asinh(mean(x)) != mean(asinh(x))`),
so we invert it back to the linear (normalized) scale with `sinh` before averaging. This only
affects the reference matrix and pseudo-bulk profiles built below; the single-cell classifier keeps
using the original `asinh`-space values it was trained on.

In [ ]:
def to_linear(X):
    return np.sinh(X)

expr_asinh = sig_adata.X
if hasattr(expr_asinh, 'toarray'):
    expr_asinh = expr_asinh.toarray()
expr_linear = to_linear(expr_asinh)

sample_labels = sig_adata.obs['sample'].values
reference_matrix = np.column_stack([
    expr_linear[ref_idx][sample_labels[ref_idx] == c].mean(axis=0)
    for c in classes
])  # (n_genes, n_classes)
print('Reference matrix shape:', reference_matrix.shape)

## 5. Simulate Pseudo-Bulk Mixtures

For each of `N_PSEUDOBULKS` samples: draw a random composition from a symmetric Dirichlet
(`DIRICHLET_ALPHA`), convert it to exact per-class cell counts via a multinomial draw (so counts
always sum to `N_CELLS_PER_BULK`), then randomly draw that many cells **with replacement** from each
class's bulk-simulation pool (bootstrap-style -- with `N_PSEUDOBULKS x N_CELLS_PER_BULK` total draws
spread over three class pools of a few thousand cells each, sampling without replacement would risk
exhausting a pool partway through). The true fraction recorded for each pseudo-bulk is the actual
realized count-based composition (`counts / N_CELLS_PER_BULK`), not the raw Dirichlet draw.

In [ ]:
pseudobulk_records = []

for b in range(N_PSEUDOBULKS):
    true_props = rng.dirichlet(np.full(len(classes), DIRICHLET_ALPHA))
    counts = rng.multinomial(N_CELLS_PER_BULK, true_props)
    true_fractions = counts / N_CELLS_PER_BULK

    drawn_indices = []
    for c, n_c in zip(classes, counts):
        if n_c == 0:
            continue
        pool = bulk_pool_class_idx[c]
        drawn_indices.append(rng.choice(pool, size=n_c, replace=True))
    drawn_indices = np.concatenate(drawn_indices) if drawn_indices else np.array([], dtype=int)

    bulk_profile_linear = expr_linear[drawn_indices].mean(axis=0)

    pseudobulk_records.append({
        'bulk_id': b,
        'true_fractions': true_fractions,
        'cell_indices': drawn_indices,
        'bulk_profile_linear': bulk_profile_linear,
    })

print(f'Simulated {len(pseudobulk_records)} pseudo-bulks of {N_CELLS_PER_BULK} cells each')

## 6. Deconvolve Each Pseudo-Bulk with NNLS

In [ ]:
for rec in pseudobulk_records:
    coeffs, _residual = nnls(reference_matrix, rec['bulk_profile_linear'])
    total = coeffs.sum()
    rec['nnls_fractions'] = coeffs / total if total > 0 else np.zeros_like(coeffs)

print('Example (first pseudo-bulk):')
print('  true:', dict(zip(classes, pseudobulk_records[0]['true_fractions'].round(3))))
print('  nnls:', dict(zip(classes, pseudobulk_records[0]['nnls_fractions'].round(3))))

## 7. Cross-Check: Aggregate Single-Cell Classifier Predictions per Pseudo-Bulk

Runs `clf_rf_sig` (the RF-only fusion-signature classifier) on the individual cells drawn into each
pseudo-bulk, then takes the empirical fraction of each predicted label -- an independent estimate of
composition that doesn't go through NNLS or the reference matrix at all.

In [ ]:
for rec in pseudobulk_records:
    cell_expr_asinh = expr_asinh[rec['cell_indices']]
    preds = clf_rf_sig.predict(cell_expr_asinh)
    counts = pd.Series(preds).value_counts()
    rec['sc_fractions'] = np.array([counts.get(c, 0) for c in classes]) / len(preds)

print('Example (first pseudo-bulk):')
print('  true:', dict(zip(classes, pseudobulk_records[0]['true_fractions'].round(3))))
print('  single-cell:', dict(zip(classes, pseudobulk_records[0]['sc_fractions'].round(3))))

## 8. Combine Results

In [ ]:
rows = []
for rec in pseudobulk_records:
    row = {'bulk_id': rec['bulk_id']}
    for i, c in enumerate(classes):
        row[f'true_{c}'] = rec['true_fractions'][i]
        row[f'nnls_{c}'] = rec['nnls_fractions'][i]
        row[f'sc_{c}'] = rec['sc_fractions'][i]
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv('pseudobulk_deconvolution_results.csv', index=False)
results_df.head()

## 9. Evaluate: True Fraction vs. Each Estimate, and the Two Estimates Against Each Other

Focused on the `fused` class since that's the fraction of specific interest, but the per-class
summary table below covers all three.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

panels = [
    (axes[0], results_df['true_fused'], results_df['nnls_fused'], 'C0',
     'True fused fraction', 'NNLS-estimated fused fraction', 'True vs. NNLS'),
    (axes[1], results_df['true_fused'], results_df['sc_fused'], 'darkorange',
     'True fused fraction', 'Single-cell-aggregated fused fraction', 'True vs. Single-Cell Classifier'),
    (axes[2], results_df['nnls_fused'], results_df['sc_fused'], 'seagreen',
     'NNLS-estimated fused fraction', 'Single-cell-aggregated fused fraction', 'NNLS vs. Single-Cell Classifier'),
]

for ax, x, y, color, xlabel, ylabel, title in panels:
    ax.scatter(x, y, alpha=0.6, s=15, color=color)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    r, _ = pearsonr(x, y)
    ax.text(
        0.05, 0.95, f'$R^2$ = {r**2:.3f}\n$r$ = {r:.3f}',
        transform=ax.transAxes, ha='left', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7, edgecolor='none'),
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

for ax in axes:
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig('pseudobulk_fused_fraction_comparison.png')
plt.show()

In [ ]:
summary_rows = []
for c in classes:
    true_c = results_df[f'true_{c}']
    nnls_c = results_df[f'nnls_{c}']
    sc_c = results_df[f'sc_{c}']

    r_nnls, _ = pearsonr(true_c, nnls_c)
    r_sc, _ = pearsonr(true_c, sc_c)
    r_cross, _ = pearsonr(nnls_c, sc_c)

    summary_rows.append({
        'class': c,
        'nnls_mae': (true_c - nnls_c).abs().mean(),
        'nnls_pearson_r': r_nnls,
        'sc_mae': (true_c - sc_c).abs().mean(),
        'sc_pearson_r': r_sc,
        'nnls_vs_sc_pearson_r': r_cross,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('pseudobulk_deconvolution_summary.csv', index=False)
summary_df